# Определение переворота текстовой строки на 180°

Решение: компактная **PP-LCNet_x1_0_textline_ori**, дообученная на синтетических русских и английских текстовых строках. Для каждого изображения возвращается вероятность `p_180`, а не только метка класса.

**Результат на тесте организаторов: 1 − Brier = 0.90386951; Brier = 0.09613049.** Этот результат получен с исходными вероятностями дообученной модели. Метки теста локально недоступны: значение сообщено организаторами после отправки CSV.

На отдельной синтетической валидации: **1 − Brier = 0.99542934**. Это другая выборка; высокий синтетический результат не является оценкой качества на реальных фотографиях. Ноутбук объясняет подход и позволяет явно запустить формирование нового submission. Он ничего не скачивает и не запускает обучение автоматически.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# Работает при запуске из корня репозитория или папки notebooks.
candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ROOT = next((p for p in candidates if (p / 'artifacts/model_card.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Откройте ноутбук внутри клонированного репозитория с artifacts/model_card.json.')

MODEL_CARD = json.loads((ROOT / 'artifacts/model_card.json').read_text(encoding='utf-8'))
VALIDATION = json.loads((ROOT / 'reports/validation/validation_metrics.json').read_text(encoding='utf-8'))
MODEL_DIR = ROOT / 'artifacts/model'
assert MODEL_CARD['model_name'] == 'PP-LCNet_x1_0_textline_ori'
print('Модель:', MODEL_CARD['model_name'])
print('Версия:', MODEL_CARD['version'])
print('Python:', sys.version.split()[0])

## Данные и честная валидация

Использованы первые RU/EN шарды `nvidia/OCR-Synthetic-Multilingual-v1`. Из синтетических страниц извлечены `line_bboxes`: по 40 000 кропов на язык, не более трёх строк с одной страницы, seed 42. После геометрического фильтра `width / height >= 1` осталось 71 155 изображений. Этот фильтр отбирает форму бокса, но сам по себе не доказывает направление текста.

Данные разделены **до** создания перевёрнутых вариантов: 60 520 исходных изображений в train и 10 635 в validation. Один `GroupShuffleSplit` по исходной странице, отдельно для каждого языка, `validation_size=0.15`, seed 42. Страницы не пересекаются между train и validation; это один holdout, не k-fold CV.

Каждый исходный кроп даёт два примера: исходный с меткой 0 и повёрнутый на 180° с меткой 1. Получаются 121 040 train-примеров и 21 270 validation-примеров с балансом классов 50/50. Перемешивание воспроизводимо, seed 42 для train и 43 для validation. Связанные ориентации одной строки всегда остаются в одной части. Исходные тестовые изображения не использованы для обучения или ручной разметки.

## Дообучение и размер входа

Старт — официальные двухклассовые веса **textline_ori**, не четырёхклассовая document-orientation модель. Архитектура и бинарная голова 0°/180° сохранены.

Параметры завершённого эксперимента: 10 эпох, batch size 64, learning rate 0.05, warmup 1 эпоха, Momentum 0.9, cosine learning-rate schedule, L2 `1e-5`, seed 42. Использованы `RandAugment` (`rand-m9-mstd0.5-inc1`) и `RandomErasing`, loss — cross-entropy. Лучший checkpoint выбран по accuracy синтетической валидации, а не по тестовому баллу; выбран epoch 10.

**Фактический вход модели: ширина 160 × высота 80 px**, тензор `[3, 80, 160]`. `ResizeImage: size: [160, 80]` означает `[W, H]`. Размер 48×384 рассматривался на этапе EDA, но в этом эксперименте не использован. RGB нормализуется с ImageNet mean/std из официального конфигурационного файла. На предсказании применяется preprocessing сохранённого экспорта; дополнительный resize вручную не нужен.

In [ ]:
# Читаем сохранённый отчёт, не выдавая синтетическую оценку за тестовую.
print('Синтетическая validation:')
for key in ['validation_samples', 'accuracy', 'brier', '1_minus_brier']:
    print(f'  {key}: {VALIDATION[key]}')
test = MODEL_CARD['test']
print('Тест организаторов (сообщённый результат):')
print('  1 − Brier:', test['user_reported_1_minus_brier'])
print('  Brier:', test['user_reported_brier'])
model_files = ['inference.json', 'inference.pdiparams', 'inference.yml']
missing = [name for name in model_files if not (MODEL_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Не хватает файлов экспорта: {missing}')
size_mib = sum((MODEL_DIR / name).stat().st_size for name in model_files) / 2**20
print(f'Размер экспорта: {size_mib:.2f} MiB')

## Вероятности, ошибки и скорость

Метрика: `Brier = mean((p_180 − y)**2)`, итоговый балл — `1 − Brier`. Уверенная ошибка штрафуется сильнее осторожной. Поэтому основной submission сохраняет вероятности. Пороговое округление `p >= 0.5 → 1`, иначе `0` исследовано отдельно и не является основной версией: на синтетической validation его балл 0.99370005 хуже исходного 0.99542934.

Для получения вероятности 180° нужен `topk=2`: код сопоставляет **per-image `label_names` с `scores`**. В использованной версии PaddleX поле `class_ids` может содержать матрицу целого batch и не подходит для такого сопоставления. Эта особенность проверена и учтена в `src/predict.py`.

Экспорт занимает примерно 6.53 MiB. Один замер на GPU 5: 21 270 синтетических примеров за 24.07 s, около 884 изображений/s, batch size 128. Измерение включает чтение и preprocessing, но не инициализацию модели. Это одиночный замер, **не** production-бенчмарк; CPU, диски, потоки и нагрузка дадут другие значения.

Разрыв между синтетической validation и тестом указывает на ограниченность переноса синтетики на реальные надписи. Дальнейшие направления: внешняя реальная валидация, короткие слова и логотипы, управляемое размытие/уменьшение разрешения и отдельные эксперименты с калибровкой. Они не заявлены как выполненные улучшения.

## Воспроизводимость

Для воспроизведения submission достаточно сохранённого экспорта в `artifacts/model`, окружения из `requirements.txt`, `sample_submission.csv` и выданных организаторами тестовых изображений. Данные передаются пользователем отдельно и не скачиваются ноутбуком. Предсказание работает локально на CPU, без внешнего inference API. Следующие две ячейки запускают `src/predict.py`; нужен явный `RUN_PREDICTION=True`. Код проверяет файлы, порядок ID и диапазон вероятностей, не перезаписывает уже существующий CSV.

Повторное обучение — отдельная операция, не необходимая для проверки предсказаний. Параметры данных находятся в `configs/data.yaml`, завершённого fine-tune — в `configs/finetune.yaml`; команды и настройка PaddleX/PaddleClas описаны в README. Например, после подготовки собственного окружения и данных: `python src/train.py train --config configs/finetune.yaml --physical-gpu-id <разрешённый_GPU>`. Notebook не устанавливает зависимости, не выбирает GPU автоматически и не запускает обучение при Run All.

## Использованные open-source источники

- [NVIDIA OCR-Synthetic-Multilingual-v1](https://huggingface.co/datasets/nvidia/OCR-Synthetic-Multilingual-v1) — исходные синтетические данные, CC BY 4.0; NVIDIA / Ryan Chesler. Использованная HF ревизия `69696a1cc543ef3a0f8e9892a89c17293e915263` восстановлена по локальным download metadata обеих языковых частей.
- [PaddleOCR: text-line orientation classification](https://www.paddleocr.ai/latest/en/version3.x/module_usage/textline_orientation_classification.html) — описание PP-LCNet textline orientation.
- [PaddleX](https://github.com/PaddlePaddle/PaddleX) и [PaddleClas](https://github.com/PaddlePaddle/PaddleClas) — архитектура, обучение и локальный inference; Apache 2.0.

Подготовка данных, групповой split, проверка вероятностей и формирование submission выполнены в коде этого репозитория. Контрольные суммы весов и исходного submission сохранены в `artifacts/model_card.json`.

In [ ]:
# Настройте только эту ячейку перед созданием submission.
RUN_PREDICTION = False
SAMPLE_PATH = ROOT / 'sample_submission.csv'
IMAGES_DIR = ROOT / 'data/test/images'
OUTPUT_PATH = ROOT / 'outputs/submission_reproduced.csv'
DEVICE = 'cpu'
BATCH_SIZE = 128
assert DEVICE == 'cpu', 'В этом ноутбуке выбран безопасный CPU-запуск; GPU настраивается отдельно через CLI.'

In [ ]:
if RUN_PREDICTION:
    required = [SAMPLE_PATH, IMAGES_DIR, ROOT / 'src/predict.py']
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError('Укажите существующие входные файлы: ' + ', '.join(missing))
    if OUTPUT_PATH.exists():
        raise FileExistsError(f'Выберите новое имя результата: {OUTPUT_PATH}')
    command = [
        sys.executable, str(ROOT / 'src/predict.py'),
        '--model-dir', str(MODEL_DIR), '--device', DEVICE,
        '--batch-size', str(BATCH_SIZE), '--sample', str(SAMPLE_PATH),
        '--images', str(IMAGES_DIR), '--output', str(OUTPUT_PATH),
    ]
    subprocess.run(command, cwd=ROOT, check=True)
    print('Готовый CSV:', OUTPUT_PATH)
else:
    print('Предсказание выключено. Укажите входные пути и RUN_PREDICTION=True для запуска.')